# edge20 — ティック新解像度: v7執行実測(docs/108)
**事前登録**: docs/108(時刻セット3つ・決定規則固定)。目的は新エッジ探索ではなく、検証済みv7/v9の執行レイヤー実測。

手順: 上から順に全セル実行(ランタイム→すべてのセルを実行)。
- セル2: Driveマウント(承認が出ます)
- セル5: ティック取得 **約5〜10分(16並列)**(途中で切れても再実行で続きから)
- セル6: 解析 → `forex_ml/tick_edge20/edge20_tick_execution.json` 出力
- 最後のJSON出力を丸ごとコピーしてClaude Codeセッションに貼り付けてください(docs/109へ転記します)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%%writefile colab_edge20_tick_fetch.py
# -*- coding: utf-8 -*-
"""
colab_edge20_tick_fetch.py — edge20(docs/108): Dukascopyティック取得(ユーザーColab用)。
v3: 起動時に接続自己診断 / 100リクエスト毎に進捗表示 / 404は即スキップ(リトライしない) /
    全滅検知で早期停止。12並列・再開可能。

使い方(Colab):
  from google.colab import drive; drive.mount('/content/drive')
  !python colab_edge20_tick_fetch.py
出力: {DRIVE_BASE}/tick_edge20/{PAIR}_m1.parquet
仕様(docs/108 §1固定): EURJPY/GBPJPY/USDJPY × 月・火曜 × 03-12時UTC × 2023-07-01..2026-06-30
"""
import os, lzma, struct, time, datetime as dt, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

DRIVE_BASE = "/content/drive/MyDrive/forex_ml"
OUT_DIR = (f"{DRIVE_BASE}/tick_edge20" if os.path.isdir("/content/drive/MyDrive")
           else os.path.join(os.path.dirname(os.path.abspath(__file__)), "data", "tick_edge20"))
PAIRS = {"EURJPY": 0.001, "GBPJPY": 0.001, "USDJPY": 0.001}
START, END = dt.date(2023, 7, 1), dt.date(2026, 6, 30)
HOURS = list(range(3, 13))
WEEKDAYS = (0, 1)
WORKERS = 12
CHECKPOINT_DAYS = 40
UA = {"User-Agent": "Mozilla/5.0"}


def url_of(pair, day, hour):
    return (f"https://datafeed.dukascopy.com/datafeed/{pair}/{day.year}/"
            f"{day.month-1:02d}/{day.day:02d}/{hour:02d}h_ticks.bi5")   # 月は0始まり


def fetch_hour(pair, day, hour, point, retries=2):
    for a in range(retries):
        try:
            req = urllib.request.Request(url_of(pair, day, hour), headers=UA)
            raw = urllib.request.urlopen(req, timeout=15).read()
            if len(raw) == 0:
                return day, "empty", []
            d = lzma.decompress(raw)
            base = dt.datetime.combine(day, dt.time(hour), tzinfo=dt.timezone.utc)
            out = [(base + dt.timedelta(milliseconds=struct.unpack(">I", d[i:i+4])[0]),
                    struct.unpack(">I", d[i+4:i+8])[0] * point,
                    struct.unpack(">I", d[i+8:i+12])[0] * point)
                   for i in range(0, len(d) - 19, 20)]
            return day, "ok", out
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return day, "404", []          # 休場等=データなし。リトライ不要
            if a == retries - 1:
                return day, f"http{e.code}", None
            time.sleep(1.0 + a)
        except Exception as e:
            if a == retries - 1:
                return day, type(e).__name__, None
            time.sleep(1.0 + a)


def selftest():
    d = dt.date(2026, 5, 4)  # 月曜
    print("[自己診断] EURJPY 2026-05-04 04hUTC を取得...")
    t0 = time.time()
    day, st, ticks = fetch_hour("EURJPY", d, 4, 0.001)
    print(f"  status={st} ticks={len(ticks) if ticks else 0} ({time.time()-t0:.1f}s)")
    if st != "ok" or not ticks:
        print("  ⚠ Dukascopyに到達できていません。Colabのネットワーク一時障害か、")
        print("    Dukascopy側の一時ブロックの可能性 → 数分待って再実行 / それでも駄目なら")
        print("    ランタイムを「切断して削除」→ 新規ランタイムで再実行してください(IPが変わります)")
        return False
    return True


def to_m1(ticks):
    df = pd.DataFrame(ticks, columns=["t", "ask", "bid"])
    df["spread_bps"] = (df["ask"] - df["bid"]) / ((df["ask"] + df["bid"]) / 2) * 1e4
    df = df.set_index("t")
    g = df.resample("1min")
    m1 = pd.DataFrame(dict(bid=g["bid"].median(), ask=g["ask"].median(),
                           spread_bps=g["spread_bps"].median(), n_ticks=g["bid"].count()))
    return m1.dropna(subset=["bid"])


def target_days():
    d, out = START, []
    while d <= END:
        if d.weekday() in WEEKDAYS:
            out.append(d)
        d += dt.timedelta(days=1)
    return out


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    if not selftest():
        return
    for pair, point in PAIRS.items():
        t0 = time.time()
        path = os.path.join(OUT_DIR, f"{pair}_m1.parquet")
        acc, done_days = [], set()
        if os.path.exists(path):
            prev = pd.read_parquet(path)
            acc = [prev]
            done_days = set(pd.Series(prev.index.date).unique())
            print(f"{pair}: 既存 {len(prev)} 行 / {len(done_days)} 日 → 続きから")
        days = [d for d in target_days() if d not in done_days]
        if not days:
            print(f"{pair}: 取得済み"); continue
        total_req = len(days) * len(HOURS)
        print(f"{pair}: {len(days)}日({total_req}リクエスト)を{WORKERS}並列で取得")
        buf = []
        stats = {"ok": 0, "404": 0, "empty": 0, "fail": 0}
        done_req = 0
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            for i0 in range(0, len(days), CHECKPOINT_DAYS):
                chunk = days[i0:i0 + CHECKPOINT_DAYS]
                futs = {ex.submit(fetch_hour, pair, d, h, point) for d in chunk for h in HOURS}
                day_ticks = {d: [] for d in chunk}
                for f in as_completed(futs):
                    day, st, ticks = f.result()
                    done_req += 1
                    if st == "ok":
                        stats["ok"] += 1; day_ticks[day].extend(ticks)
                    elif st in ("404", "empty"):
                        stats[st] += 1
                    else:
                        stats["fail"] += 1
                    if done_req % 100 == 0:
                        rate = done_req / max(time.time() - t0, 1)
                        eta = (total_req - done_req) / max(rate, 0.1)
                        print(f"  {pair}: {done_req}/{total_req} ok={stats['ok']} 404={stats['404']} "
                              f"fail={stats['fail']} ({rate:.0f}req/s 残り目安{eta/60:.0f}分)")
                    if done_req == 60 and stats["ok"] == 0:
                        print("  ⚠ 60リクエスト全滅 → 中断。自己診断の案内に従って再実行してください")
                        for g in futs:
                            g.cancel()
                        return
                for d in chunk:
                    if day_ticks[d]:
                        buf.append(to_m1(sorted(day_ticks[d])))
                pd.concat(acc + buf).sort_index().to_parquet(path)
                print(f"  {pair}: {chunk[-1]} まで保存 ({i0+len(chunk)}/{len(days)}日, {time.time()-t0:.0f}s)")
        print(f"{pair}: 完了 {time.time()-t0:.0f}s ok={stats['ok']} 404={stats['404']} fail={stats['fail']} → {path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile edge20_tick_execution.py
# -*- coding: utf-8 -*-
"""
edge20_tick_execution.py — edge20(docs/108)の解析ハーネス。colab_edge20_tick_fetch.py の
出力(1分足集約parquet)から H20a/H20b/H20c を採点する。Colab/ローカル両対応。

出力: edge20_tick_execution.json(Drive優先、なければ research/results/)→ docs/109 へ転記。
規律: 時刻セットは事前登録の3つのみ。リターン系の新パターン探索はしない。
"""
import os, json, datetime as dt
import numpy as np, pandas as pd

DRIVE_BASE = "/content/drive/MyDrive/forex_ml"
IN_DIR = (f"{DRIVE_BASE}/tick_edge20" if os.path.isdir("/content/drive/MyDrive")
          else os.path.join(os.path.dirname(os.path.abspath(__file__)), "data", "tick_edge20"))
OUT_PATH = (f"{DRIVE_BASE}/tick_edge20/edge20_tick_execution.json"
            if os.path.isdir("/content/drive/MyDrive")
            else os.path.join(os.path.dirname(os.path.abspath(__file__)), "results", "edge20_tick_execution.json"))
PAIRS = ["EURJPY", "GBPJPY", "USDJPY"]
PIP = 0.01
HOUR_SETS = {"base_04061810": [4, 6, 8, 10],      # 現行(基準)
             "alt_03050709": [3, 5, 7, 9],
             "alt_05070911": [5, 7, 9, 11]}
BLOCKS = [("Y1", "2023-07-01", "2024-06-30"), ("Y2", "2024-07-01", "2025-06-30"),
          ("Y3", "2025-07-01", "2026-06-30")]


def load(pair):
    df = pd.read_parquet(os.path.join(IN_DIR, f"{pair}_m1.parquet")).sort_index()
    idx = pd.to_datetime(df.index)
    if idx.tz is None:
        idx = idx.tz_localize("UTC")
    df.index = idx
    df["pips"] = (df["ask"] - df["bid"]) / PIP
    return df


def h20a(dfs):
    out = {}
    for pair, df in dfs.items():
        mon = df[df.index.weekday == 0]
        byh = {}
        for h in range(3, 13):
            s = mon[mon.index.hour == h]["pips"].dropna()
            if len(s) == 0:
                continue
            byh[h] = dict(median_pips=round(float(s.median()), 2),
                          p90=round(float(s.quantile(.90)), 2),
                          p95=round(float(s.quantile(.95)), 2), n_min=int(len(s)))
        out[pair] = byh
    return out


def first_minute_price(df, day, hour, col):
    w = df[(df.index.date == day) & (df.index.hour == hour)]
    return float(w[col].iloc[0]) if len(w) else None


def h20b(dfs):
    res = {}
    for name, hours in HOUR_SETS.items():
        per_block = {}
        for bname, b0, b1 in BLOCKS:
            tot = 0.0; nshots = 0
            for pair, df in dfs.items():
                sub = df[(df.index >= pd.Timestamp(b0, tz="UTC")) & (df.index <= pd.Timestamp(b1, tz="UTC") + pd.Timedelta(days=1))]
                mondays = sorted(set(sub[sub.index.weekday == 0].index.date))
                for d in mondays:
                    d2 = d + dt.timedelta(days=1)
                    for h in hours:
                        e = first_minute_price(sub, d, h, "ask")
                        x = first_minute_price(sub, d2, h, "bid")
                        if e and x and e > 0:
                            tot += (x / e - 1.0); nshots += 1
            per_block[bname] = dict(net_pct=round(tot * 100, 3), shots=nshots)
        res[name] = per_block
    # 決定規則1: 代替が3ブロック全てで基準を上回る場合のみ提案
    base = res["base_04061810"]
    winner = None
    for name in ("alt_03050709", "alt_05070911"):
        if all(res[name][b]["net_pct"] > base[b]["net_pct"] for b, _, _ in BLOCKS):
            winner = name
            break
    return res, winner


def h20c(dfs):
    out = {}
    for pair, df in dfs.items():
        mon = df[(df.index.weekday == 0) & (df.index.hour.isin(HOUR_SETS["base_04061810"]))]
        s = mon["pips"].dropna()
        p95 = float(s.quantile(.95))
        out[pair] = dict(entry_hours_median_pips=round(float(s.median()), 2),
                         p95=round(p95, 2),
                         proposed_MaxSpreadPips=round(p95 * 1.2, 1),
                         pct_minutes_over_3pips=round(float((s > 3.0).mean() * 100), 2))
    return out


def main():
    dfs = {}
    for p in PAIRS:
        f = os.path.join(IN_DIR, f"{p}_m1.parquet")
        if not os.path.exists(f):
            print(f"⚠ {f} なし — 先に colab_edge20_tick_fetch.py を実行"); return
        dfs[p] = load(p)
        print(f"{p}: {len(dfs[p])} 分足行 {dfs[p].index.min()}..{dfs[p].index.max()}")

    a = h20a(dfs)
    b, winner = h20b(dfs)
    c = h20c(dfs)
    # 決定規則3: 実測往復コスト(スプレッド中央値×2側面≒往復) vs 検証仮定(≈2pips往復)
    med_entry = np.mean([c[p]["entry_hours_median_pips"] for p in PAIRS])
    cost_flag = bool(med_entry * 1.0 > 2.0 * 2)   # 中央スプレッド(片側)が仮定往復2pipsの2倍相当を超えるか

    out = dict(prereg="docs/108",
               h20a_spread_by_hour=a,
               h20b_hourset_compare=b,
               h20b_winner=(winner or "変更なし(基準維持)"),
               h20c_spread_cap=c,
               cost_reinterpretation_needed=cost_flag,
               note="時刻セット変更/上限変更は提案止まり。実装は承認+デモ後(docs/108 §3)")
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    with open(OUT_PATH, "w") as f:
        json.dump(out, f, ensure_ascii=False, indent=1, default=str)
    print(json.dumps(out, ensure_ascii=False, indent=1)[:2000])
    print("保存:", OUT_PATH, "→ docs/109 へ転記")


if __name__ == "__main__":
    main()


In [ ]:
!python colab_edge20_tick_fetch.py


In [ ]:
!python edge20_tick_execution.py


実行後: 上のセルの末尾に表示されたJSON(または Drive `forex_ml/tick_edge20/edge20_tick_execution.json`)をコピーして、Claude Code セッションへ貼り付け。